# Objectives

> Planning costs of rolled-out trajectories, joint `(B, S)` or per agent `(B, S, num_agents)`.

In [ ]:
#| default_exp planning.objective

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
# Adapted from stable-worldmodel's planning package (https://github.com/galilai-group/stable-worldmodel):
#
# MIT License
#
# Copyright (c) 2026 GalilAI-group
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F

from stable_marl.protocols import Objective

In [ ]:
#| export
def _reduce(err: torch.Tensor, reduction: str, per_agent: bool, agent_dim: int) -> torch.Tensor:
    "Collapse every axis after the candidate one (keeping `agent_dim` if `per_agent`)."
    dims = tuple(d for d in range(2, err.ndim) if not (per_agent and d == agent_dim))
    return err.sum(dim=dims) if reduction == 'sum' else err.mean(dim=dims)


STEP_REDUCTIONS = ('last', 'min', 'mean')


class GoalMSE(nn.Module):
    """
    Squared error between the predicted embeddings ``predicted_emb`` ``(B, S, T, ...)`` and the
    last goal embedding ``goal_emb`` ``(B, T_goal, ...)``.

    `step_reduction` chooses which predicted steps count: ``'last'`` (the end of the plan, as in
    stable-worldmodel), or over the predicted future (the last ``horizon`` steps, ``horizon`` read
    from ``action_candidates``) its ``'min'`` (the plan's best match, e.g. reaching the goal
    mid-plan) or ``'mean'``.

    Returns ``(B, S)``, or ``(B, S, num_agents)`` with ``per_agent=True``, where the agent axis
    of the embeddings is `agent_dim` (3: ``(B, S, T, num_agents, ...)``).
    """
    def __init__(self, pred_key: str = 'predicted_emb', goal_key: str = 'goal_emb', reduction: str = 'sum',
                 per_agent: bool = False, agent_dim: int = 3, step_reduction: str = 'last'):
        super().__init__()
        if reduction not in ('sum', 'mean'):
            raise ValueError(f"reduction must be 'sum' or 'mean', got {reduction!r}")
        if step_reduction not in STEP_REDUCTIONS:
            raise ValueError(f"step_reduction must be one of {STEP_REDUCTIONS}, got {step_reduction!r}")
        self.pred_key, self.goal_key, self.reduction = pred_key, goal_key, reduction
        self.per_agent, self.agent_dim, self.step_reduction = per_agent, agent_dim, step_reduction

    def forward(self, info_dict: dict) -> torch.Tensor:
        pred = info_dict[self.pred_key]
        if self.step_reduction == 'last':
            steps = pred[:, :, -1:]
        else:
            candidates = info_dict.get('action_candidates')
            steps = pred[:, :, -candidates.shape[2]:] if candidates is not None else pred
        goal = info_dict[self.goal_key][:, None, -1:].expand_as(steps)
        err = F.mse_loss(steps, goal.detach(), reduction='none')
        # per step: every axis after time (but the agent axis if per_agent), then over the steps
        dims = tuple(d for d in range(3, err.ndim) if not (self.per_agent and d == self.agent_dim))
        per_step = err.sum(dim=dims) if self.reduction == 'sum' else err.mean(dim=dims)
        if self.step_reduction == 'min':
            return per_step.min(dim=2).values
        return per_step.mean(dim=2) if self.step_reduction == 'mean' else per_step[:, :, 0]


class ControlPenalty(nn.Module):
    """
    Squared norm of the action candidates ``(B, S, horizon, num_agents, ...)``: ``(B, S)``, or
    ``(B, S, num_agents)`` with ``per_agent=True``.
    """
    def __init__(self, action_key: str = 'action_candidates', per_agent: bool = False, agent_dim: int = 3):
        super().__init__()
        self.action_key, self.per_agent, self.agent_dim = action_key, per_agent, agent_dim

    def forward(self, info_dict: dict) -> torch.Tensor:
        return _reduce(info_dict[self.action_key].pow(2), 'sum', self.per_agent, self.agent_dim)


class WeightedSum(nn.Module):
    "Linear combination ``sum(w * term(info_dict))`` of objectives (all joint or all per agent)."
    def __init__(self, terms: list[tuple[float, Objective]]):
        super().__init__()
        if not terms:
            raise ValueError("WeightedSum requires at least one term")
        self.weights = [weight for weight, _ in terms]
        self.objectives = nn.ModuleList(term for _, term in terms)

    def forward(self, info_dict: dict) -> torch.Tensor:
        return sum(weight * term(info_dict) for weight, term in zip(self.weights, self.objectives))

### Tests

In [ ]:
pred = torch.zeros(2, 5, 4, 3, 2)               # (B, S, T, agents, dim)
pred[:, :, -1, 1] = 1.0                         # agent 1 ends 1 away from the goal on each axis
info = {'predicted_emb': pred, 'goal_emb': torch.zeros(2, 1, 3, 2), 'action_candidates': torch.ones(2, 5, 4, 3, 7)}
assert GoalMSE()(info).shape == (2, 5) and torch.allclose(GoalMSE()(info), torch.full((2, 5), 2.0))
per_agent = GoalMSE(per_agent=True)(info)
assert per_agent.shape == (2, 5, 3) and per_agent[0, 0].tolist() == [0.0, 2.0, 0.0]
assert ControlPenalty(per_agent=True)(info).shape == (2, 5, 3)
assert torch.allclose(WeightedSum([(1.0, GoalMSE()), (0.5, ControlPenalty())])(info), torch.full((2, 5), 2.0 + 0.5 * 84))

# scoring every predicted step: an agent passing through the goal mid-plan
pred = torch.ones(1, 1, 5, 2, 2)                     # context (1 step) + horizon 4, 2 agents
pred[0, 0, 2, 0] = 0.0                               # agent 0 at the goal at the 2nd predicted step only
info = {'predicted_emb': pred, 'goal_emb': torch.zeros(1, 1, 2, 2), 'action_candidates': torch.zeros(1, 1, 4, 2, 4)}
assert GoalMSE(per_agent=True)(info)[0, 0].tolist() == [2.0, 2.0]
assert GoalMSE(per_agent=True, step_reduction='min')(info)[0, 0].tolist() == [0.0, 2.0]
assert GoalMSE(per_agent=True, step_reduction='mean')(info)[0, 0].tolist() == [1.5, 2.0]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()